In [ ]:
import os as _os
import sys as _sys

_root = _os.path.abspath(_os.getcwd())
while _root != _os.path.dirname(_root) and not _os.path.isfile(_os.path.join(_root, "eval_metrics.py")):
    _root = _os.path.dirname(_root)
if _root not in _sys.path:
    _sys.path.insert(0, _root)
from eval_metrics import evaluate, summary, time_inference, time_inference_torch
from gps_ids import protocol
from gps_ids.results import append_validated

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, f1_score
import os
import gc
import time

In [ ]:
# Constants
RANDOM_STATE = 1082
MODEL_NAME = "StackingClassifier(LogisticRegression RandomForest SVC)"

SYNTH_GENERATORS = ["ctgan", "gc", "tvae"]
SAMPLE_SIZES = [10_000, 100_000, 1_000_000]
SYNTH_FILES = [f"spoofing_{g}_{n}.csv" for g in SYNTH_GENERATORS for n in SAMPLE_SIZES]


In [5]:
# Define base models
base_learners = [
    ('lr', LogisticRegression(random_state=RANDOM_STATE, max_iter=1200, solver='saga')),
    ('rf', RandomForestClassifier(random_state=RANDOM_STATE)),
    ('svc', SVC(probability=True, random_state=RANDOM_STATE))
]

# Meta model
meta_learner = LogisticRegression(random_state=RANDOM_STATE, max_iter=1200, solver='saga')

In [ ]:
import gc

results = []
for synth_name in SYNTH_FILES:
    try:
        EXP = protocol.synthetic_vs_real_heldout("2d", synth_name)
    except FileNotFoundError:
        print(f"Skipping {synth_name}: file not found")
        continue
    print(f"\nProcessing {synth_name}")

    stacking_model = StackingClassifier(
        estimators=base_learners,
        final_estimator=meta_learner,
        passthrough=False,
        n_jobs=-1
    )
    start_time = time.time()
    stacking_model.fit(EXP.X_train, EXP.y_train)
    train_time = time.time() - start_time

    y_pred = stacking_model.predict(EXP.X_test)
    accuracy = accuracy_score(EXP.y_test, y_pred)
    f1 = f1_score(EXP.y_test, y_pred, average='weighted', zero_division=0)
    print(f"Training on synthetic: {synth_name}, testing on real data")
    print(classification_report(EXP.y_test, y_pred, zero_division=0))
    print(f"F1 Score: {f1:.4f}")
    print(f"Accuracy: {accuracy:.4f}")

    _row = evaluate(
        EXP.y_test,
        y_pred,
        model_name=MODEL_NAME,
        family="Synthetic-Stacking",
        y_proba=stacking_model.predict_proba(EXP.X_test) if hasattr(stacking_model, "predict_proba") else None,
        train_time_sec=train_time,
        num_epochs="N/A",
        parameters=str(stacking_model.get_params()),
        timing=time_inference(stacking_model.predict, EXP.X_test),
        **EXP.eval_kwargs(),
    )
    print(summary(_row))
    results.append(_row)
    del EXP, y_pred
    gc.collect()


In [ ]:
from gps_ids.schema import results_path

append_validated(results, str(results_path("2d")))
print(f"Results appended to {results_path('2d')}")
